# Lesson 03: The decoder

The encoder (Lesson 02) turned the image into features `[f1 … f5]`: `f5` knows **what** is there, but only at a tiny resolution.
The **decoder** turns them back into a full-size mask that says **where** the object is.

```
f5 [1024, 22]  ─ up ─┐
                     merge ◄── f4 [512, 44]   (skip connection)
                     conv
                     ─ up ─┐
                           merge ◄── f3 [256, 88]
                           conv
                           ─ up ─┐
                                 merge ◄── f2 [128, 176]
                                 conv
                                 ─ up ─┐
                                       merge ◄── f1 [64, 352]
                                       conv
                                       head (1×1 conv) ──► mask [1, 352, 352]
```

**The decoder contract:**

> `decoder([f1, f2, f3, f4, f5])` → **mask logits** `[B, 1, H, W]`

Every decoder step has three parts, and **each one is a knob you can modify**:

| step | options | who uses what |
|---|---|---|
| **up** (upsample ×2) | bilinear, nearest, `ConvTranspose2d` | UNet paper: transpose; most modern code: bilinear |
| **merge** (skip connection) | concat, add, (attention, later) | UNet: concat · FPN / many COD models: add |
| **conv** (refine) | `DoubleConv`, residual, … | same bricks as the encoder |

| Part | Topic |
|---|---|
| A | Upsampling: bilinear vs nearest vs `ConvTranspose2d` (and its checkerboard problem) |
| B | Why skip connections? Measured on your COD10K masks |
| C | `DecoderBlock` and `UNetDecoder` |
| D | The knobs: concat vs add, bilinear vs transpose, width, which skips |
| E | Sanity check: train encoder + decoder on **one** COD10K image and watch it learn |

## Setup

In [ ]:
import time

import cv2
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from blocks import ConvBNReLU, DoubleConv, count_params, show, to_tensor
from config import IMAGE_SIZE, cod10k_pairs, get_device
from encoders import ResNet50Encoder, UNetEncoder

device = get_device()
print(f"device: {device}")

test_pairs = cod10k_pairs("Test")
print(f"COD10K test images: {len(test_pairs)}")


def load_pair(index):
    """Return (image BGR, mask 0/255) resized to IMAGE_SIZE, or a synthetic pair if COD10K is missing."""
    if test_pairs:
        image_path, mask_path = test_pairs[index]
        image = cv2.resize(cv2.imread(image_path), (IMAGE_SIZE, IMAGE_SIZE))
        mask = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), (IMAGE_SIZE, IMAGE_SIZE),
                          interpolation=cv2.INTER_NEAREST)
        return image, mask
    rng = np.random.default_rng(index)
    image = cv2.add(np.full((IMAGE_SIZE, IMAGE_SIZE, 3), (60, 120, 80), np.uint8),
                    rng.integers(0, 40, (IMAGE_SIZE, IMAGE_SIZE, 3), dtype=np.uint8))
    mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    pts = (rng.integers(60, 290, (7, 2))).astype(np.int32)
    cv2.fillPoly(mask, [cv2.convexHull(pts)], 255)
    image[mask > 0] = cv2.add(image[mask > 0], np.array([10, 15, 10], np.uint8))
    return image, mask


def iou(pred, gt):
    """Intersection over Union of two boolean masks (1.0 = perfect)."""
    inter = np.logical_and(pred, gt).sum()
    union = np.logical_or(pred, gt).sum()
    return inter / union if union > 0 else 1.0


SAMPLE_INDEX = 0  # ✏️ change to look at other images
image, mask = load_pair(SAMPLE_INDEX)
show([image, mask], ["image", "ground truth"], cols=2)

## Part A: upsampling, three ways

Each decoder step has to double H and W. There are three common ways:

| method | learnable? | how |
|---|---|---|
| **nearest** | no | copy each pixel into a 2×2 block (blocky) |
| **bilinear** | no | weighted average of the 4 nearest pixels (smooth) |
| **`ConvTranspose2d`** | **yes** | a "reverse convolution" with learned weights |

You already know the first two from OpenCV: `cv2.resize(..., interpolation=cv2.INTER_NEAREST / INTER_LINEAR)`.
`F.interpolate` is the PyTorch version, and it runs on the GPU inside the network.

Below we shrink the GT mask to 22×22 (the size of the UNet encoder's `f5`) and blow it back up.

In [ ]:
m = torch.from_numpy(mask).float()[None, None] / 255.0        # [1, 1, 352, 352]
small = F.interpolate(m, size=(22, 22), mode="area")          # like cv2.INTER_AREA

up_nearest = F.interpolate(small, size=(IMAGE_SIZE, IMAGE_SIZE), mode="nearest")
up_bilinear = F.interpolate(small, size=(IMAGE_SIZE, IMAGE_SIZE), mode="bilinear", align_corners=False)

# The OpenCV equivalent gives (almost) the same result:
cv_bilinear = cv2.resize(small[0, 0].numpy(), (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
print(f"max difference F.interpolate vs cv2.resize (bilinear): {np.abs(cv_bilinear - up_bilinear[0, 0].numpy()).max():.4f}")

show([mask, small[0, 0].numpy(), up_nearest[0, 0].numpy(), up_bilinear[0, 0].numpy()],
     ["GT 352x352", "shrunk to 22x22", "nearest up", "bilinear up"], cols=4)

### `ConvTranspose2d` and the checkerboard problem

A transposed convolution spreads each input pixel onto a k×k patch of the output, placed `stride` pixels apart.
If `kernel_size` is **not divisible by** `stride`, the patches overlap **unevenly**: some output pixels get 1 contribution, others 2 or 4.
The result is the famous **checkerboard artifact**.

Here we set all weights to 1 and feed in a flat image of ones, so any pattern in the output comes from the layer itself.

In [ ]:
flat = torch.ones(1, 1, 8, 8)

configs = {
    "k=3, s=2  (uneven)": dict(kernel_size=3, stride=2, padding=1, output_padding=1),
    "k=2, s=2  (no overlap)": dict(kernel_size=2, stride=2),
    "k=4, s=2  (even overlap)": dict(kernel_size=4, stride=2, padding=1),
}
outs, titles = [], []
for name, cfg in configs.items():
    tconv = nn.ConvTranspose2d(1, 1, bias=False, **cfg)
    nn.init.ones_(tconv.weight)
    with torch.no_grad():
        out = tconv(flat)[0, 0]
    print(f"{name:<26} 8x8 -> {tuple(out.shape)}   unique values: {sorted(set(out[2:-2, 2:-2].flatten().tolist()))}")
    outs.append(out.numpy())
    titles.append(name)

show(outs, titles, cmap="viridis", cols=3)

In the real network the weights are learned, so the pattern is weaker, but it often still shows up as a faint grid in the predicted masks.
That's why most modern segmentation code uses **bilinear upsampling followed by a conv**, and the UNet decoder here does the same by default.

> ✏️ **TRY IT**
> - Change `size=(22, 22)` to `(11, 11)` (ResNet50's `f5`). How much of the fish's shape survives?
> - Try `mode="bicubic"` in `F.interpolate`.

## Part B: why skip connections?

Suppose the deepest feature were **perfect**: it knows exactly where the object is, but only at 22×22 or 11×11.
Upsample that perfect answer back to 352×352 and compare it with the GT. The IoU you get is the **best possible** result without any finer information.

We measure this over many COD10K test masks.

In [ ]:
N = min(200, len(test_pairs)) if test_pairs else 20
resolutions = [176, 88, 44, 22, 11]
scores = {r: [] for r in resolutions}
for i in range(N):
    _, gt = load_pair(i)
    gt_bool = gt > 127
    if gt_bool.sum() == 0:
        continue
    for r in resolutions:
        small = cv2.resize(gt, (r, r), interpolation=cv2.INTER_AREA)
        back = cv2.resize(small, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR)
        scores[r].append(iou(back > 127, gt_bool))

print(f"Best possible IoU from a PERFECT low-resolution mask ({N} images):")
for r in resolutions:
    stride = IMAGE_SIZE // r
    s = np.array(scores[r])
    print(f"  {r:>3}x{r:<3} (stride {stride:>2}):  mean IoU {s.mean():.3f}   worst {s.min():.3f}")

In [ ]:
views = [mask]
for r in [44, 22, 11]:
    small = cv2.resize(mask, (r, r), interpolation=cv2.INTER_AREA)
    views.append(cv2.resize(small, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_LINEAR) > 127)
show([v.astype(np.uint8) * 255 if v.dtype == bool else v for v in views],
     ["GT", "from 44x44", "from 22x22", "from 11x11"], cols=4)

**The lesson:** even a perfect `f5` loses the thin parts (fins, legs, antennae), and those are exactly the hard parts in COD.
The **skip connections** bring back the fine detail from `f1`–`f3`, which still know where the edges are.
That's what the `merge` step does.

> ✏️ **TRY IT**
> - Look for the worst images: sort by `scores[11]` and display them with `load_pair(i)`. What shape are those objects?

## Part C: `DecoderBlock` and `UNetDecoder`

**One `DecoderBlock`** = `up` → `merge skip` → `DoubleConv`

With `merge="concat"`, the channels stack: `in_ch + skip_ch`, and the conv reduces them to `out_ch`.
With `merge="add"`, the skip first goes through a 1×1 conv so that its channels match, and the two are summed.

In [ ]:
def make_up(kind, ch):
    """The upsampling step: H, W -> 2H, 2W."""
    if kind == "transpose":
        return nn.ConvTranspose2d(ch, ch, kernel_size=2, stride=2)  # learnable
    if kind == "nearest":
        return nn.Upsample(scale_factor=2, mode="nearest")
    if kind == "bilinear":
        return nn.Upsample(scale_factor=2, mode="bilinear", align_corners=False)
    raise ValueError(kind)


class DecoderBlock(nn.Module):
    """
    One decoder step:  upsample  ->  merge the skip feature  ->  conv.

    in_ch   : channels coming up from the deeper decoder step
    skip_ch : channels of the encoder feature at this scale (0 = no skip)
    out_ch  : channels this block outputs
    up      : "bilinear" | "nearest" | "transpose"
    merge   : "concat" (UNet)  |  "add" (FPN / many COD models)
    """

    def __init__(self, in_ch, skip_ch, out_ch, up="bilinear", merge="concat"):
        super().__init__()
        self.up = make_up(up, in_ch)
        self.merge = merge if skip_ch > 0 else None
        if self.merge == "concat":
            self.fuse = DoubleConv(in_ch + skip_ch, out_ch)      # channels stack up
        elif self.merge == "add":
            self.skip_proj = ConvBNReLU(skip_ch, in_ch, kernel_size=1)  # match channels first
            self.fuse = DoubleConv(in_ch, out_ch)
        else:
            self.fuse = DoubleConv(in_ch, out_ch)                # no skip at all

    def forward(self, x, skip=None):
        x = self.up(x)
        if self.merge is None or skip is None:
            return self.fuse(x)
        if x.shape[-2:] != skip.shape[-2:]:  # e.g. odd input sizes
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        if self.merge == "concat":
            x = torch.cat([x, skip], dim=1)
        else:
            x = x + self.skip_proj(skip)
        return self.fuse(x)

In [ ]:
# One block on its own: f5 (1024 ch, 22x22) + skip f4 (512 ch, 44x44)
f5 = torch.randn(1, 1024, 22, 22)
f4 = torch.randn(1, 512, 44, 44)

for merge in ["concat", "add"]:
    block = DecoderBlock(1024, 512, 512, merge=merge)
    out = block(f5, f4)
    print(f"merge={merge:<7}: f5 {tuple(f5.shape)} + f4 {tuple(f4.shape)} -> {tuple(out.shape)}   params {count_params(block):,}")

**The full decoder** stacks one block per skip, from deep to shallow, and ends with a **head**:
a 1×1 conv down to 1 channel. The output is a **logit** (any real number). `torch.sigmoid(logit)` turns it into a 0–1 probability that the pixel is the object.

`SegModel` just plugs any encoder into the decoder.

In [ ]:
class UNetDecoder(nn.Module):
    """
    Walks back up the encoder features, deepest first:

        f5 -> block(+f4) -> block(+f3) -> block(+f2) -> block(+f1) -> head -> mask logits

    encoder_channels : the encoder's `.channels`, e.g. [64, 128, 256, 512, 1024]
    decoder_channels : output channels of each block, deep -> shallow (default: mirror the encoder)
    use_skips        : one True/False per block, deep -> shallow
    """

    def __init__(self, encoder_channels, decoder_channels=None, up="bilinear", merge="concat",
                 use_skips=None, num_classes=1):
        super().__init__()
        skip_channels = list(encoder_channels[:-1])[::-1]        # [f4, f3, f2, f1] channels
        decoder_channels = list(decoder_channels or skip_channels)
        use_skips = list(use_skips or [True] * len(skip_channels))
        self.use_skips = use_skips

        blocks = []
        in_ch = encoder_channels[-1]                              # start from f5
        for skip_ch, out_ch, use in zip(skip_channels, decoder_channels, use_skips):
            blocks.append(DecoderBlock(in_ch, skip_ch if use else 0, out_ch, up=up, merge=merge))
            in_ch = out_ch
        self.blocks = nn.ModuleList(blocks)
        self.head = nn.Conv2d(in_ch, num_classes, kernel_size=1)  # 1 channel = "object" score

    def forward(self, feats, out_size=None):
        x = feats[-1]
        skips = feats[-2::-1]                                     # [f4, f3, f2, f1]
        for block, skip, use in zip(self.blocks, skips, self.use_skips):
            x = block(x, skip if use else None)
        x = self.head(x)
        if out_size is not None and x.shape[-2:] != tuple(out_size):
            # ResNet's f1 is stride 2, so the last step still needs a 2x upsample
            x = F.interpolate(x, size=out_size, mode="bilinear", align_corners=False)
        return x  # logits: apply torch.sigmoid() to get a 0..1 mask


class SegModel(nn.Module):
    """Any encoder + any decoder = a segmentation model."""

    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder

    def forward(self, x):
        return self.decoder(self.encoder(x), out_size=x.shape[-2:])

In [ ]:
def describe_model(model, x):
    model.eval()
    with torch.no_grad():
        feats = model.encoder(x)
        out = model.decoder(feats, out_size=x.shape[-2:])
    print("encoder features:", [tuple(f.shape[1:]) for f in feats])
    print(f"output logits   : {tuple(out.shape)}")
    print(f"params -> encoder {count_params(model.encoder):,}   decoder {count_params(model.decoder):,}   "
          f"total {count_params(model):,}")


x = to_tensor(image).to(device)

print("=== UNet encoder + UNet decoder (the original UNet) ===")
enc = UNetEncoder()
unet = SegModel(enc, UNetDecoder(enc.channels)).to(device)
describe_model(unet, x)

print("\n=== ResNet50 encoder + UNet decoder ===")
enc = ResNet50Encoder(pretrained=False)  # weights don't matter for counting; Part E uses pretrained
res_unet = SegModel(enc, UNetDecoder(enc.channels, decoder_channels=(256, 128, 64, 32))).to(device)
describe_model(res_unet, x)

Notice that the **same decoder class** works with both encoders, because both follow the contract from Lesson 02 (a list of 5 features plus `.channels`).

For ResNet50 we chose a **narrower** decoder `(256, 128, 64, 32)`. If it mirrored the encoder (1024, 512, 256, 64), it would be huge.
COD models do the same: SINet reduces every feature to just **32 channels** before decoding.

## Part D: the knobs

Same ResNet50 encoder each time, different decoders. Compare the decoder params and check that the output is always `[1, 1, 352, 352]`.

In [ ]:
enc = ResNet50Encoder(pretrained=False).to(device).eval()
with torch.no_grad():
    feats = enc(x)

DECODER_VARIANTS = {
    "concat + bilinear (default)": dict(decoder_channels=(256, 128, 64, 32)),
    "add + bilinear":              dict(decoder_channels=(256, 128, 64, 32), merge="add"),
    "concat + transpose":          dict(decoder_channels=(256, 128, 64, 32), up="transpose"),
    "narrow (64,64,32,32)":        dict(decoder_channels=(64, 64, 32, 32)),
    "wide (mirror encoder)":       dict(),
    "no skips at all":             dict(decoder_channels=(256, 128, 64, 32), use_skips=(False,) * 4),
    "skip f1 (like SINet)":        dict(decoder_channels=(256, 128, 64, 32), use_skips=(True, True, True, False)),
}

print(f"{'decoder variant':<30}{'decoder params':>15}   output")
for name, cfg in DECODER_VARIANTS.items():
    dec = UNetDecoder(enc.channels, **cfg).to(device).eval()
    with torch.no_grad():
        out = dec(feats, out_size=x.shape[-2:])
    print(f"{name:<30}{count_params(dec):>15,}   {tuple(out.shape)}")

**Which knob does what**

| knob | effect |
|---|---|
| `merge="concat"` | keeps encoder and decoder features separate, and the conv learns how to mix them. More params. |
| `merge="add"` | cheaper; assumes both features mean "the same kind of thing" after the 1×1 projection. |
| `up="transpose"` | learnable upsampling, but watch for the checkerboard (Part A). It's also expensive on deep features: on ResNet's 2048-channel `f5` it alone is 2048×2048×2×2 ≈ 16.8M weights, which is why the decoder nearly triples. |
| `decoder_channels` | the biggest lever on decoder size and memory. |
| `use_skips` | which scales feed in. Dropping `f1` saves a lot of memory (it's the biggest map); SINet decodes from `f2`–`f5` only. |

> ✏️ **TRY IT**
> - Add your own variant: `merge="add", up="transpose", decoder_channels=(128, 64, 32, 16)`.
> - Why does "no skips at all" still output a full-size mask? What information is it missing? (Part B)

## Part E: sanity check, can it learn one image?

Before training on 3040 images (Lesson 07), professionals always check that the model can **memorise a single image**.
If it can't, something is broken (the shapes, the loss, or the learning rate). It only takes about 30 seconds.

- **Model:** pretrained ResNet50 encoder + UNet decoder
- **Loss:** `BCEWithLogitsLoss`, binary cross-entropy: "is each pixel object or background?"
- **Optimizer:** Adam
- **Mixed precision** (`autocast`), as in Lesson 02 Part E

> Memorising one image is **not** real training: it will not work on other images. It only proves the pipeline is correct.

In [ ]:
torch.manual_seed(0)
STEPS = 100
SNAPSHOTS = [0, 10, 30, 60, STEPS - 1]

enc = ResNet50Encoder(pretrained=True)
model = SegModel(enc, UNetDecoder(enc.channels, decoder_channels=(256, 128, 64, 32))).to(device)
model.train()

x = to_tensor(image).to(device)
y = torch.from_numpy((mask > 127).astype(np.float32))[None, None].to(device)  # target: 0 or 1

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=3e-4)
use_amp = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

snapshots, start = {}, time.perf_counter()
for step in range(STEPS):
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        logits = model(x)
        loss = criterion(logits.float(), y)
    scaler.scale(loss).backward()
    scaler.step(optimizer)
    scaler.update()

    if step in SNAPSHOTS:
        prob = torch.sigmoid(logits.detach().float())[0, 0].cpu().numpy()
        snapshots[step] = prob
        print(f"step {step:>3}   loss {loss.item():.4f}   IoU {iou(prob > 0.5, mask > 127):.3f}")

print(f"{STEPS} steps in {time.perf_counter() - start:.1f} s")

In [ ]:
show([image, mask] + [snapshots[s] for s in SNAPSHOTS],
     ["image", "GT"] + [f"step {s}" for s in SNAPSHOTS], cmap="gray", cols=4, size=3.5)

You should see the loss drop and the prediction change from grey noise into the fish's shape, with IoU heading towards 0.9+.
If it does: the encoder, the decoder, the loss and the optimizer are all wired correctly. 🎉

> ✏️ **TRY IT** (re-run the two cells above after each change)
> - `use_skips=(False,) * 4` in the decoder: does it still reach the same IoU? Look at the thin parts of the mask.
> - `merge="add"` or `up="transpose"`: faster or slower to learn?
> - `ResNet50Encoder(pretrained=False)`: how much slower is it without pretrained weights?
> - `lr=1e-2` or `lr=1e-6`: what does a learning rate that's too high or too low look like?

---
## Summary

- A decoder step is **up → merge skip → conv**, repeated until you are back at full size, then a **1×1 head** outputs the mask logits.
- **Skips** carry the fine detail that the deep features lose (Part B). For thin camouflaged parts, they're essential.
- The parts you can modify: the **upsampling method**, the **merge** (concat/add), the **width**, and **which skips** are used.
- Encoder and decoder only share the contract (a list of features + channels), so you can mix and match them freely.

**Next lesson (04): UNet end to end**: a proper training loop on COD10K with train/val splits, metrics and saving the best model.
(Later: UNet++ replaces the simple skips with nested ones, and SINet replaces the plain merge with search and attention modules.)